# Inspect Protected Subspace Spectrum

Analyze the eigenvalue spectrum of layer activation covariances.
Use this to:
- Verify that the spectrum decays rapidly (concentration hypothesis)
- Choose the right k for the protected subspace
- Confirm that low-rank protection is theoretically justified

Run after `scripts/05_build_subspaces.sh`.

In [ ]:
import json
import sys
from pathlib import Path

import torch
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.subspace.analyze import plot_singular_value_decay, plot_explained_variance_vs_k

MODEL_SHORT = 'Llama-3.1-8B-Instruct'
SUBSPACES_DIR = ROOT / f'artifacts/subspaces/{MODEL_SHORT}_protected_mix'
print('Subspaces dir:', SUBSPACES_DIR)
print('Exists:', SUBSPACES_DIR.exists())

## 1. Explained Variance vs k (All Layers)

In [ ]:
summary_path = SUBSPACES_DIR / 'subspace_summary.json'
assert summary_path.exists(), f'Run script 05 first. Not found: {summary_path}'

summary = json.loads(summary_path.read_text())
print(f'Layers: {len(summary)}')
print('Layer names:', list(summary.keys())[:3], '...')

fig = plot_explained_variance_vs_k(
    summary=summary,
    highlight_layers=None,  # set specific layer names to bold-highlight
)
plt.show()

## 2. Eigenvalue Spectrum for a Single Layer

In [ ]:
# Pick a layer to inspect
layer_name = list(summary.keys())[0]  # change to inspect a specific layer
print('Inspecting layer:', layer_name)

# Load the largest-k subspace artifact
ks = sorted(int(k) for k in summary[layer_name].keys())
max_k = ks[-1]
stem = f"{layer_name.replace('.', '_')}_k{max_k}"
pt_path = SUBSPACES_DIR / 'layers' / f'{stem}.pt'

data = torch.load(pt_path, map_location='cpu')
lambda_k = data['lambda_k']
print(f'Loaded {len(lambda_k)} eigenvalues')

fig = plot_singular_value_decay(
    eigenvalues=lambda_k,
    layer_name=layer_name,
    top_n=min(128, len(lambda_k)),
)
plt.show()

## 3. Choose k — Cumulative Explained Variance Table

In [ ]:
import pandas as pd

# Build a table: for each k, show explained variance across all layers
k_values = sorted(set(
    int(k) for layer_vals in summary.values() for k in layer_vals.keys()
))

rows = []
for k in k_values:
    evs = [v.get(str(k)) for v in summary.values() if str(k) in v]
    rows.append({
        'k': k,
        'mean_explained_variance': np.mean(evs) if evs else None,
        'min_explained_variance': np.min(evs) if evs else None,
        'layers_at_90pct': sum(1 for v in evs if v is not None and v >= 0.90),
        'layers_at_95pct': sum(1 for v in evs if v is not None and v >= 0.95),
        'n_layers': len(evs),
    })

df_k = pd.DataFrame(rows)
print(df_k.to_string(index=False))

## 4. Spectral Tail Analysis (Theorem 1 bound)

In [ ]:
# Compute Tr(Σ_l^{perp}) for each k, which bounds the uncontrolled interference term.
# A fast decay = small tail = theorem bound is tight.

print('Spectral tail analysis for layer:', layer_name)
print()

total_variance = lambda_k.sum().item()
print(f'Total variance (Tr Σ): {total_variance:.4f}')
print()
print(f'{'k':>6} | {'protected':>12} | tail (Tr Σ_perp):>16 | {'tail/total':>10}')
print('-' * 55)
for k in k_values:
    protected = lambda_k[:k].sum().item()
    tail = total_variance - protected
    print(f'{k:>6} | {protected:>12.4f} | {tail:>18.4f} | {tail/total_variance:>10.4f}')